# Jableh CA-Markov -- Full Pipeline

End-to-end notebook: data acquisition -> statistics -> transition matrix ->
Markov projection -> pixel-level spatial analysis -> uncertainty ->
hindcasting -> CA allocation (with slope/road) -> policy scenarios ->
cartographic exports.

Run top to bottom in Google Colab with a GEE-enabled account. Each section
is also available as a standalone, testable function in `jableh_ca_markov/`
-- this notebook wires them together against the real Jableh raster stack.

## 1. Setup: install package + dependencies

In [ ]:
!pip install -q git+https://github.com/yamenetoo/RremoteSensing.git#subdirectory=lulc_toolkit
!pip install -q rasterio geopandas shapely richdem scikit-learn cartopy
!apt-get install -y -qq unrar

# Install the jableh_ca_markov package itself (editable, from this repo)
!pip install -q -e .


## 2. Authenticate Earth Engine + load Jableh boundary

In [ ]:
import ee
from lulc_toolkit.auth import authenticate_ee
from lulc_toolkit.shapefile_utils import (
    mount_drive_readonly, find_archive_in_drive,
    extract_shapefile_archive, load_shapefile,
)

PROJECT_ID = "friendly-magpie-439921-a8"
ARCHIVE_NAME = "Jableh_Extracted.rar"

authenticate_ee(PROJECT_ID)
mount_drive_readonly()
archive_path = find_archive_in_drive(ARCHIVE_NAME)
shp_path = extract_shapefile_archive(archive_path, "/content/shapefile_data")
gdf = load_shapefile(shp_path)
geom = ee.Geometry(gdf.geometry.iloc[0].__geo_interface__)
print(f"Loaded Jableh boundary with {len(gdf)} feature(s).")


## 3. Download Dynamic World rasters (2015-2026)

In [ ]:
from lulc_toolkit.dynamic_world import download_dynamicworld_for_years

YEARS = list(range(2015, 2027))
OUTPUT_BASE = "/content/DynamicWorld_Output"

files, skipped = download_dynamicworld_for_years(
    gdf=gdf, name_field=None, years=YEARS, out_dir=OUTPUT_BASE,
)
print(f"Downloaded {len(files)} rasters; skipped: {skipped}")


## 4. Compute annual statistics + convert to styled PNG maps

In [ ]:
from jableh_ca_markov.mapping import convert_rasters_to_png
from lulc_toolkit.statistics import build_statistics_table, build_pivot_table
from lulc_toolkit.dynamic_world import DW_CLASS_NAMES

def compute_statistics(input_dir, stats_csv, pivot_csv, class_names=DW_CLASS_NAMES):
    stats_df = build_statistics_table(input_dir, class_names)
    stats_df.to_csv(stats_csv)
    pivot_df = build_pivot_table(stats_df)
    pivot_df.to_csv(pivot_csv)
    return stats_df, pivot_df

stats_df, pivot_df = compute_statistics(
    input_dir=OUTPUT_BASE,
    stats_csv="/content/DynamicWorld_Stats/DynamicWorld_Statistics.csv",
    pivot_csv="/content/DynamicWorld_Stats/Pivot_km2.csv",
)

png_files = convert_rasters_to_png(
    input_dir=OUTPUT_BASE, output_dir="/content/DynamicWorld_Maps_PNG",
)


## 5. Transition matrix, stationarity test, Markov projection, steady-state
(offline-capable -- uses only the area time series from Step 4)

In [ ]:
from jableh_ca_markov.pipeline import load_pivot_csv, run_statistical_pipeline

areas = load_pivot_csv("/content/DynamicWorld_Stats/Pivot_km2.csv")
results = run_statistical_pipeline(
    areas, horizon_years=4, n_bootstrap=1000,
    output_dir="/content/drive/MyDrive/Jableh/CA_Markov_Outputs",
)

print("P_annual:\n", results["P_annual"])
print("\nStationarity (pooled p-value):", results["stationarity"]["pooled_p"])
print("\nMarkov projection 2026-2030:\n", results["projection"])
print("\nSteady-state:\n", results["steady_state"])
print("\nSeed-matrix sensitivity:\n", results["seed_sensitivity"])
print("\nForest-protection scenario:\n", results["forest_protection_projection"])


## 6. Load raster stack into memory (needed for all pixel-level steps below)

In [ ]:
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

TARGET_CRS = "EPSG:32636"

def reproject_to_utm(src):
    transform, width, height = calculate_default_transform(
        src.crs, TARGET_CRS, src.width, src.height, *src.bounds)
    dest = np.zeros((height, width), dtype=src.dtypes[0])
    reproject(source=rasterio.band(src, 1), destination=dest,
              src_transform=src.transform, src_crs=src.crs,
              dst_transform=transform, dst_crs=TARGET_CRS,
              resampling=Resampling.nearest)
    return dest, transform

rasters, transform = {}, None
for year in YEARS:
    with rasterio.open(f"{OUTPUT_BASE}/DW_Jablah_{year}.tif") as src:
        if src.crs.is_geographic:
            data, t = reproject_to_utm(src)
        else:
            data, t = src.read(1), src.transform
    rasters[year] = data
    if transform is None:
        transform = t

pixel_area_km2 = abs(transform.a * transform.e) / 1e6
pixel_size_m = transform.a
print(f"Loaded {len(rasters)} rasters. Pixel area: {pixel_area_km2*1e6:.4f} m^2")


## 7. Pixel-level transition matrix (2015 vs 2026) + comparison with IPF/RAS

In [ ]:
from jableh_ca_markov.markov_ipf import pixel_level_transition_matrix, compare_pixel_and_ipf_matrices
from jableh_ca_markov.projection import matrix_power_to_span
from jableh_ca_markov.config import CLASS_INDICES

pixel_result = pixel_level_transition_matrix(
    rasters[2015], rasters[2026], class_indices=CLASS_INDICES, pixel_area_km2=pixel_area_km2,
)
print("Direct pixel-level transition PROBABILITY matrix (2015 -> 2026):")
print(pixel_result["probability"].round(4))

# Compare against the IPF/RAS matrix raised to the same 11-year span
P_11yr = matrix_power_to_span(results["P_annual"], n_years=11)
import pandas as pd
from jableh_ca_markov.config import CLASS_NAMES
P_11yr_df = pd.DataFrame(P_11yr, index=CLASS_NAMES, columns=CLASS_NAMES)

comparison = compare_pixel_and_ipf_matrices(pixel_result["probability"], P_11yr_df)
print("\nComparison (persistence, pixel-level vs IPF/RAS-estimated):")
print(comparison.round(4))


## 8. Direction / hotspot analysis (sector, centroid, SDE, KDE)

In [ ]:
from jableh_ca_markov.spatial_analysis import (
    class_transition_mask, sector_growth_series, sector_growth_rates,
    centroid_series, centroid_shift_metres, standard_deviational_ellipse,
    hotspot_analysis,
)
from jableh_ca_markov.config import FOREST, CROPS, BUILT

forest_to_built = class_transition_mask(rasters[2015], rasters[2026], FOREST, BUILT)
crops_to_built = class_transition_mask(rasters[2015], rasters[2026], CROPS, BUILT)
print(f"Forest->Built: {forest_to_built.sum()} px, {forest_to_built.sum()*pixel_area_km2:.2f} km^2")
print(f"Crops->Built:  {crops_to_built.sum()} px, {crops_to_built.sum()*pixel_area_km2:.2f} km^2")

built_masks = {y: (rasters[y] == BUILT).astype(np.uint8) for y in YEARS}
sector_df = sector_growth_series(built_masks, pixel_area_km2)
print("\nSector growth rates (km^2/yr):")
print(sector_growth_rates(sector_df).round(3))

centroids = centroid_series(built_masks)
shift = centroid_shift_metres(centroids, transform.a, transform.e)
print("\nCentroid shift 2015->2026:", shift)

sde_2015 = standard_deviational_ellipse(built_masks[2015])
sde_2026 = standard_deviational_ellipse(built_masks[2026])
print("\nSDE 2015:", sde_2015)
print("SDE 2026:", sde_2026)

built_gain = ((built_masks[2026] == 1) & (built_masks[2015] == 0)).astype(np.uint8)
hotspot = hotspot_analysis(built_gain, sigma_px=30, percentile=95)
print(f"\nHotspot: {hotspot['gain_pixels']} gain px, "
      f"{hotspot['hotspot_pixels']} hotspot px "
      f"({hotspot['hotspot_pixels']*pixel_area_km2:.2f} km^2)")


## 9. Slope + road-distance layers (revision item 2.3)

In [ ]:
from jableh_ca_markov.ca_allocation import load_and_normalise_slope, load_and_normalise_road_distance

DEM_PATH = "/content/drive/MyDrive/Jableh/srtm_jableh.tif"          # SRTM 30m DEM, user-provided
ROADS_PATH = "/content/drive/MyDrive/Jableh/osm_roads_jableh.gpkg"  # OSM roads, user-provided

slope_degrees = load_and_normalise_slope(
    DEM_PATH, target_shape=rasters[2026].shape, target_transform=transform, target_crs=TARGET_CRS,
)
road_distance_m = load_and_normalise_road_distance(
    ROADS_PATH, target_shape=rasters[2026].shape, target_transform=transform,
    target_crs=TARGET_CRS, pixel_size_m=pixel_size_m,
)
print("Slope (degrees) range:", slope_degrees.min(), "-", slope_degrees.max())
print("Road distance (m) range:", road_distance_m.min(), "-", road_distance_m.max())


## 10. Spatially explicit CA allocation for 2030 (with slope + road)

In [ ]:
from jableh_ca_markov.ca_allocation import run_ca_allocation

conv_to_built = results["P_annual"][:, CLASS_NAMES.index("built")]
built_targets = {
    int(year): float(results["projection"].loc[year, "built"])
    for year in [2027, 2028, 2029, 2030]
}

ca_result = run_ca_allocation(
    lulc_base=rasters[2026],
    built_targets_by_year=built_targets,
    pixel_area_km2=pixel_area_km2,
    pixel_size_m=pixel_size_m,
    conv_to_built=conv_to_built,
    D0=500.0, window=5,
    slope_degrees=slope_degrees,
    road_distance_m=road_distance_m,
)
print("Allocation log:")
for entry in ca_result["allocation_log"]:
    print(" ", entry)

final_built_km2 = (ca_result["final_lulc"] == BUILT).sum() * pixel_area_km2
print(f"\nFinal simulated 2030 built-up area: {final_built_km2:.2f} km^2")


## 11. Retrospective hindcasting (2015-2022 -> 2023-2026), OA / Kappa / FoM

In [ ]:
from jableh_ca_markov.validation import run_hindcast
from jableh_ca_markov.config import build_seed_matrix

def ca_allocation_wrapper(lulc_base, built_targets_by_year, **kwargs):
    return run_ca_allocation(
        lulc_base=lulc_base, built_targets_by_year=built_targets_by_year,
        pixel_area_km2=pixel_area_km2, pixel_size_m=pixel_size_m,
        conv_to_built=conv_to_built, D0=500.0, window=5,
        slope_degrees=slope_degrees, road_distance_m=road_distance_m,
    )

hindcast_report = run_hindcast(
    areas_by_year=areas,
    observed_rasters_by_year=rasters,
    lulc_2022=rasters[2022],
    class_indices=CLASS_INDICES,
    seed_matrix=build_seed_matrix(),
    ca_allocation_fn=ca_allocation_wrapper,
)
print(hindcast_report)
hindcast_report.to_csv("/content/drive/MyDrive/Jableh/CA_Markov_Outputs/hindcast_validation.csv")


## 12. Monte Carlo uncertainty (model) -- already computed in Step 5
## Classification-error perturbation (data uncertainty, revision 4.1)

In [ ]:
# Plug in the REAL Dynamic World confusion matrix (Brown et al. 2022
# supplementary accuracy tables) here before running -- placeholder
# identity-like matrix shown for illustration only.
import pandas as pd
from jableh_ca_markov.uncertainty import bootstrap_with_perturbation

combined_uncertainty = bootstrap_with_perturbation(
    results["pair_matrices"], areas[2026], horizon_years=4,
    perturbation_std_frac=0.05, n_bootstrap=100,
)
print(combined_uncertainty["summary"])


## 13. Policy scenarios: green belt, forest protection, combined

In [ ]:
from jableh_ca_markov.scenarios import (
    build_southern_buffer_mask, run_forest_protection_scenario, compare_scenarios,
)

from shapely.geometry import shape
jableh_shape = shape(gdf.geometry.iloc[0].__geo_interface__)

green_belt_mask = build_southern_buffer_mask(
    jableh_shape, buffer_m=500, transform=transform, crs=TARGET_CRS,
    target_shape=rasters[2026].shape,
)

forest_protection_proj = run_forest_protection_scenario(
    results["P_annual"], areas[2026], 2026, 4, new_trees_to_built=0.01,
)

scenario_table = compare_scenarios(
    baseline_projection=results["projection"],
    green_belt_projection=None,  # pixel-level green-belt CA run wired similarly to Step 10
    forest_protection_projection=forest_protection_proj,
    combined_projection=forest_protection_proj,  # placeholder until green-belt CA run is added
    target_year=2030,
)
print(scenario_table)


## 14. Cartographic exports: Sentinel-2 basemap + Syria locator + detail-with-inset

In [ ]:
from jableh_ca_markov.mapping import export_sentinel2_basemap, locator_map_syria, detail_map_with_inset

basemap_png = export_sentinel2_basemap(
    geom, output_tif_local_path="/content/Jableh_HighRes_Basemap.png",
)
locator_png = locator_map_syria(
    jableh_shape, output_path="/content/Jableh_Locator_Map_Syria.png",
)
detail_png = detail_map_with_inset(
    jableh_shape, output_path="/content/Jableh_Syria_Locator_Detail_Map.png",
)
print("Saved:", basemap_png, locator_png, detail_png)


## 15. Zip and download all results

In [ ]:
import glob, os, zipfile

png_files_all = glob.glob("/content/**/*.png", recursive=True)
zip_path = "/content/jableh_all_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in png_files_all:
        zf.write(f, os.path.relpath(f, "/content/"))

from google.colab import files as colab_files
colab_files.download(zip_path)
